# aule — Climate metrics

This notebook covers `aule.metrics.climate`: metrics that operate along the **time axis**, designed for climate model validation use cases (seasonal cycles, extremes, trends, temporal persistence). These metrics require a meaningful time dimension, so we always pass `data_format='hwct'` to indicate the array shape is `(H, W, C, T)`.

In [ ]:
!pip install aule

## Setup

We simulate a 100-time-step field with a seasonal-like oscillation plus a slow warming trend, then add noise for the "prediction".

In [ ]:
import numpy as np

np.random.seed(1)

T = 100
t = np.arange(T)
seasonal_signal = np.sin(2 * np.pi * t / 20)        # fast oscillation, like a seasonal cycle
trend_signal    = 0.01 * t                          # slow linear warming trend

base = (seasonal_signal + trend_signal).reshape(1, 1, 1, T)
gt   = np.tile(base, (32, 32, 1, 1)) + np.random.normal(0, 0.05, (32, 32, 1, T))
pred = gt + np.random.normal(0, 0.1, gt.shape)

print('gt shape:', gt.shape)  # (H, W, C, T)

## Seasonal error

`seasonal_error` compares the spatial-mean time series of ground truth vs prediction (i.e. how well the model captures the temporal evolution, ignoring spatial detail).

In [ ]:
from aule.metrics import seasonal_error

score = seasonal_error(gt, pred, data_format='hwct')
print('Seasonal error (MSE of spatial-mean series):', score)

## Percentile error

Useful for checking whether a model captures extremes well, even if average behavior looks fine. For example, P95 captures heatwave-like extremes; P5 captures cold extremes.

In [ ]:
from aule.metrics import percentile_error

print('P95 error:', percentile_error(gt, pred, percentile=95.0))
print('P5 error: ', percentile_error(gt, pred, percentile=5.0))

## Pixel-wise temporal correlation

`pixelwise_temporal_correlation` computes, at every pixel, the Pearson correlation across time between ground truth and prediction — giving a correlation *map* rather than a single number. Useful to spot regions where temporal dynamics are poorly captured.

In [ ]:
from aule.metrics import pixelwise_temporal_correlation

r_map = pixelwise_temporal_correlation(gt, pred, data_format='hwct')
print('r_map shape:', r_map.shape)  # (H, W, C)
print('Mean correlation:', r_map.mean())
print('Min correlation: ', r_map.min())

## Trend error

`trend_error` fits a linear trend (least squares) to the spatial-mean time series of both ground truth and prediction, then compares the slopes. This isolates whether the model gets the *long-term trend* right, independent of short-term variability.

In [ ]:
from aule.metrics import trend_error

score = trend_error(gt, pred, data_format='hwct')
print('Trend slope error:', score)

## Extreme event duration error

`extreme_event_duration_error` measures the mean duration of "events" (consecutive time steps above/below a threshold, e.g. heatwaves or cold spells) and compares ground truth vs prediction.

In [ ]:
from aule.metrics import extreme_event_duration_error

threshold = 0.8
score = extreme_event_duration_error(gt, pred, threshold=threshold, above=True, data_format='hwct')
print(f'Mean event-duration error (threshold={threshold}):', score)

## Autocorrelation error

`autocorrelation_error` compares the temporal autocorrelation function (ACF) up to a maximum lag — relevant for checking whether a model preserves temporal persistence (e.g. drought persistence, slow climate modes like ENSO).

In [ ]:
from aule.metrics import autocorrelation_error

score = autocorrelation_error(gt, pred, max_lag=10, data_format='hwct')
print('Mean ACF error (lags 1-10):', score)

## Visualizing the spatial-mean time series

A quick plot helps interpret the metrics above: we can see the seasonal cycle, the slow trend, and how closely the (noisy) prediction follows the ground truth.

In [ ]:
import matplotlib.pyplot as plt

gt_mean = gt.mean(axis=(0, 1, 2))
pred_mean = pred.mean(axis=(0, 1, 2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, gt_mean, label='Ground truth (spatial mean)', color='black')
ax.plot(t, pred_mean, label='Prediction (spatial mean)', color='#D65F5F', ls='--')
ax.axhline(threshold, color='gray', ls=':', label=f'Extreme event threshold ({threshold})')
ax.set_xlabel('Time step')
ax.set_ylabel('Value')
ax.legend()
plt.show()